 # Análisis Exploratorio de Datos - Compras Públicas Ecuador (2015-2025)

 ## Objetivo
 Realizar un análisis completo de las compras públicas en Ecuador, identificando
 patrones, tendencias y relaciones entre variables clave.

 **Autor:** [Tu Nombre Completo]
 **Fecha:** Octubre 2025

 ## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente ✓")

Librerías importadas correctamente ✓


 ## 2. Carga de Datos

 Para este análisis, utilizaremos datos del Sistema Nacional de Contratación Pública (SERCOP).
 Los datos pueden provenir de:
 - API del SERCOP: https://datosabiertos.compraspublicas.gob.ec/
 - Archivo CSV descargado

 En este ejemplo, generaremos datos simulados realistas para demostración.

In [ ]:
def generar_datos_simulados(n_registros=5000):
    """Genera datos simulados de compras públicas"""
    np.random.seed(42)
    
    años = np.random.choice(range(2015, 2026), n_registros)
    
    data = {
        'año': años,
        'mes': np.random.randint(1, 13, n_registros),
        'provincia': np.random.choice([
            'Pichincha', 'Guayas', 'Azuay', 'Manabí', 'El Oro', 
            'Los Ríos', 'Tungurahua', 'Esmeraldas', 'Imbabura', 'Loja'
        ], n_registros),
        'tipo_compra': np.random.choice([
            'Subasta Inversa', 'Menor Cuantía', 'Licitación', 
            'Contratación Directa', 'Catálogo Electrónico'
        ], n_registros),
        'sector': np.random.choice([
            'Salud', 'Educación', 'Infraestructura', 'Seguridad',
            'Servicios Públicos', 'Tecnología', 'Transporte'
        ], n_registros),
        'monto_usd': np.random.lognormal(9, 2, n_registros),
        'estado': np.random.choice([
            'Finalizado', 'En proceso', 'Cancelado', 'Adjudicado'
        ], n_registros, p=[0.6, 0.2, 0.1, 0.1]),
        'num_oferentes': np.random.randint(1, 15, n_registros),
        'tiempo_proceso_dias': np.random.randint(15, 180, n_registros)
    }
    
    df = pd.DataFrame(data)
    df['fecha'] = pd.to_datetime(
        df['año'].astype(str) + '-' + df['mes'].astype(str) + '-01'
    )
    
    return df

# Cargar datos
df_raw = generar_datos_simulados(5000)
print(f"Datos cargados: {len(df_raw)} registros")
print(f"Período: {df_raw['año'].min()} - {df_raw['año'].max()}")

Datos cargados: 5000 registros
Período: 2015 - 2025


In [ ]:
# Visualizar primeras filas
df_raw.head(10)

,año,mes,provincia,tipo_compra,sector,monto_usd,estado,num_oferentes,tiempo_proceso_dias,fecha
0,2021,1,Loja,Contratación Directa,Servicios Públicos,8948.914573,Finalizado,1,41,2021-01-01
1,2018,12,Los Ríos,Licitación,Infraestructura,5243.246877,En proceso,4,138,2018-12-01
2,2025,9,Esmeraldas,Menor Cuantía,Tecnología,2679.653715,Finalizado,3,100,2025-09-01
3,2022,7,Imbabura,Subasta Inversa,Infraestructura,689.030124,Finalizado,6,161,2022-07-01
4,2019,2,Los Ríos,Subasta Inversa,Seguridad,2388.617882,En proceso,6,42,2019-02-01
5,2021,6,Imbabura,Catálogo Electrónico,Educación,41542.539857,Finalizado,1,165,2021-06-01
6,2024,2,Manabí,Catálogo Electrónico,Seguridad,9760.838158,Finalizado,14,78,2024-02-01
7,2017,1,Los Ríos,Licitación,Infraestructura,31292.189098,En proceso,14,63,2017-01-01
8,2021,7,Esmeraldas,Licitación,Infraestructura,730.092954,Cancelado,11,162,2021-07-01
9,2025,2,Imbabura,Subasta Inversa,Transporte,145551.701817,Adjudicado,2,125,2025-02-01


In [ ]:
# Información del dataset
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   año                  5000 non-null   int64         
 1   mes                  5000 non-null   int32         
 2   provincia            5000 non-null   object        
 3   tipo_compra          5000 non-null   object        
 4   sector               5000 non-null   object        
 5   monto_usd            5000 non-null   float64       
 6   estado               5000 non-null   object        
 7   num_oferentes        5000 non-null   int32         
 8   tiempo_proceso_dias  5000 non-null   int32         
 9   fecha                5000 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int32(3), int64(1), object(4)
memory usage: 332.2+ KB


 ## 3. Limpieza y Preprocesamiento de Datos

In [ ]:
def limpiar_datos(df):
    """Limpieza y normalización del dataset"""
    df_clean = df.copy()
    
    print("Iniciando limpieza de datos...")
    print(f"Registros iniciales: {len(df_clean)}")
    
    # Eliminar duplicados
    duplicados_antes = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"Duplicados eliminados: {duplicados_antes - len(df_clean)}")
    
    # Manejar valores nulos
    nulos_antes = df_clean.isnull().sum().sum()
    df_clean = df_clean.dropna(subset=['monto_usd', 'año'])
    print(f"Valores nulos eliminados: {nulos_antes}")
    
    # Estandarizar textos
    df_clean['provincia'] = df_clean['provincia'].str.strip().str.title()
    df_clean['tipo_compra'] = df_clean['tipo_compra'].str.strip()
    df_clean['sector'] = df_clean['sector'].str.strip()
    
    # Eliminar valores negativos o cero en montos
    df_clean = df_clean[df_clean['monto_usd'] > 0]
    
    # Filtrar años válidos (2015-2025)
    df_clean = df_clean[(df_clean['año'] >= 2015) & (df_clean['año'] <= 2025)]
    
    print(f"Registros finales: {len(df_clean)}")
    print("Limpieza completada ✓")
    
    return df_clean

df = limpiar_datos(df_raw)

Iniciando limpieza de datos...
Registros iniciales: 5000
Duplicados eliminados: 0
Valores nulos eliminados: 0
Registros finales: 5000
Limpieza completada ✓


In [ ]:
# Verificar datos limpios
df.describe()

,año,mes,monto_usd,num_oferentes,tiempo_proceso_dias,fecha
count,5000.000000,5000.000000,5.000000e+03,5000.000000,5000.00000,5000
mean,2019.938800,6.518800,6.236440e+04,7.606200,96.63500,2020-05-25 07:27:33.120000
min,2015.000000,1.000000,5.817820e+00,1.000000,15.00000,2015-01-01 00:00:00
25%,2017.000000,4.000000,2.144328e+03,4.000000,55.00000,2017-09-01 00:00:00
50%,2020.000000,7.000000,8.521132e+03,8.000000,98.00000,2020-06-01 00:00:00
75%,2023.000000,10.000000,3.323949e+04,11.000000,138.25000,2023-02-08 00:00:00
max,2025.000000,12.000000,1.547877e+07,14.000000,179.00000,2025-12-01 00:00:00
std,3.155859,3.454451,3.948242e+05,4.003489,47.98212,NaN


 ## 4. Análisis Exploratorio de Datos (EDA)

 ### 4.1 KPIs Principales

In [ ]:
# Calcular KPIs
total_contratos = len(df)
monto_total = df['monto_usd'].sum()
monto_promedio = df['monto_usd'].mean()
monto_mediano = df['monto_usd'].median()
tiempo_promedio = df['tiempo_proceso_dias'].mean()

print("=" * 60)
print("KPIs PRINCIPALES")
print("=" * 60)
print(f"Total de Contratos:          {total_contratos:,}")
print(f"Monto Total (USD):           ${monto_total:,.2f}")
print(f"Monto Promedio (USD):        ${monto_promedio:,.2f}")
print(f"Monto Mediano (USD):         ${monto_mediano:,.2f}")
print(f"Tiempo Promedio (días):      {tiempo_promedio:.0f}")
print("=" * 60)

KPIs PRINCIPALES
Total de Contratos:          5,000
Monto Total (USD):           $311,821,990.90
Monto Promedio (USD):        $62,364.40
Monto Mediano (USD):         $8,521.13
Tiempo Promedio (días):      97


 ### 4.2 Análisis Temporal

In [ ]:
# Contratos por año
contratos_año = df.groupby('año').agg({
    'monto_usd': ['sum', 'mean', 'count']
}).reset_index()
contratos_año.columns = ['año', 'monto_total', 'monto_promedio', 'cantidad']

print("\nContratos por Año:")
print(contratos_año)


Contratos por Año:
     año   monto_total  monto_promedio  cantidad
0   2015  3.008187e+07    61769.746569       487
1   2016  3.059193e+07    70326.278429       435
2   2017  2.186007e+07    46510.786290       470
3   2018  3.979417e+07    89024.993823       447
4   2019  2.163928e+07    46237.783668       468
5   2020  2.849210e+07    61010.928085       467
6   2021  1.502082e+07    34451.416450       436
7   2022  2.077312e+07    44387.009162       468
8   2023  4.753318e+07   106100.849663       448
9   2024  3.012970e+07    67707.193209       445
10  2025  2.590575e+07    60386.353722       429


In [ ]:
# Gráfico: Número de contratos por año
fig1 = px.bar(
    contratos_año,
    x='año',
    y='cantidad',
    title='Evolución del Número de Contratos por Año (2015-2025)',
    labels={'año': 'Año', 'cantidad': 'Cantidad de Contratos'},
    color='cantidad',
    color_continuous_scale='Blues',
    text='cantidad'
)
fig1.update_traces(texttemplate='%{text:,}', textposition='outside')
fig1.update_layout(height=500, showlegend=False)
fig1.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Gráfico: Monto total por año
fig2 = px.line(
    contratos_año,
    x='año',
    y='monto_total',
    title='Evolución del Monto Total Contratado por Año (USD)',
    labels={'año': 'Año', 'monto_total': 'Monto Total (USD)'},
    markers=True
)
fig2.update_traces(line=dict(width=3), marker=dict(size=10))
fig2.update_layout(height=500)
fig2.show()

: 

In [ ]:
# Análisis mensual
df['mes_nombre'] = pd.to_datetime(df['mes'], format='%m').dt.month_name()
contratos_mes = df.groupby('mes_nombre').size().reset_index(name='cantidad')

meses_orden = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
contratos_mes['mes_nombre'] = pd.Categorical(
    contratos_mes['mes_nombre'], 
    categories=meses_orden, 
    ordered=True
)
contratos_mes = contratos_mes.sort_values('mes_nombre')

fig3 = px.bar(
    contratos_mes,
    x='mes_nombre',
    y='cantidad',
    title='Distribución de Contratos por Mes del Año',
    labels={'mes_nombre': 'Mes', 'cantidad': 'Cantidad'},
    color='cantidad',
    color_continuous_scale='Viridis'
)
fig3.update_layout(height=500, xaxis_tickangle=-45)
fig3.show()

: 

 ### 4.3 Análisis Geográfico

In [ ]:
# Análisis por provincia
provincias_stats = df.groupby('provincia').agg({
    'monto_usd': ['sum', 'mean', 'count']
}).reset_index()
provincias_stats.columns = ['provincia', 'monto_total', 'monto_promedio', 'cantidad']
provincias_stats = provincias_stats.sort_values('monto_total', ascending=False)

print("\nTop 10 Provincias por Monto Total:")
print(provincias_stats.head(10))

: 

In [ ]:
# Gráfico de pastel: Top 10 provincias por cantidad de contratos
top10_prov = df.groupby('provincia').size().reset_index(name='cantidad')
top10_prov = top10_prov.sort_values('cantidad', ascending=False).head(10)

fig4 = px.pie(
    top10_prov,
    values='cantidad',
    names='provincia',
    title='Top 10 Provincias por Número de Contratos',
    hole=0.4
)
fig4.update_traces(textposition='inside', textinfo='percent+label')
fig4.update_layout(height=600)
fig4.show()

: 

In [ ]:
# Gráfico de barras horizontales: Top 10 provincias por monto
top10_monto = provincias_stats.head(10).sort_values('monto_total', ascending=True)

fig5 = px.bar(
    top10_monto,
    x='monto_total',
    y='provincia',
    orientation='h',
    title='Top 10 Provincias por Monto Total Contratado (USD)',
    labels={'monto_total': 'Monto Total (USD)', 'provincia': 'Provincia'},
    color='monto_total',
    color_continuous_scale='Reds',
    text='monto_total'
)
fig5.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig5.update_layout(height=500, showlegend=False)
fig5.show()

: 

 ### 4.4 Análisis por Sector y Tipo de Compra

In [ ]:
# Análisis por sector
sector_stats = df.groupby('sector').agg({
    'monto_usd': ['sum', 'mean', 'count']
}).reset_index()
sector_stats.columns = ['sector', 'monto_total', 'monto_promedio', 'cantidad']
sector_stats = sector_stats.sort_values('monto_total', ascending=False)

print("\nEstadísticas por Sector:")
print(sector_stats)

: 

In [ ]:
# Gráfico: Contratos por sector y año
sector_año = df.groupby(['año', 'sector']).size().reset_index(name='cantidad')

fig6 = px.bar(
    sector_año,
    x='año',
    y='cantidad',
    color='sector',
    title='Evolución de Contratos por Sector (2015-2025)',
    labels={'año': 'Año', 'cantidad': 'Cantidad de Contratos', 'sector': 'Sector'},
    barmode='group'
)
fig6.update_layout(height=600, legend=dict(orientation="v", yanchor="top", y=1, xanchor="right", x=1))
fig6.show()

: 

In [ ]:
# Análisis por tipo de compra
tipo_compra_stats = df.groupby('tipo_compra').agg({
    'monto_usd': ['sum', 'mean', 'count']
}).reset_index()
tipo_compra_stats.columns = ['tipo_compra', 'monto_total', 'monto_promedio', 'cantidad']
tipo_compra_stats = tipo_compra_stats.sort_values('monto_total', ascending=False)

print("\nEstadísticas por Tipo de Compra:")
print(tipo_compra_stats)

: 

In [ ]:
# Gráfico: Monto por tipo de compra
fig7 = px.bar(
    tipo_compra_stats,
    x='tipo_compra',
    y='monto_total',
    title='Monto Total por Tipo de Compra (USD)',
    labels={'tipo_compra': 'Tipo de Compra', 'monto_total': 'Monto Total (USD)'},
    color='monto_total',
    color_continuous_scale='Greens',
    text='monto_total'
)
fig7.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig7.update_layout(height=500, xaxis_tickangle=-45, showlegend=False)
fig7.show()

: 

In [ ]:
# Distribución por estado
estado_dist = df.groupby('estado').size().reset_index(name='cantidad')

fig8 = px.pie(
    estado_dist,
    values='cantidad',
    names='estado',
    title='Distribución de Contratos por Estado',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig8.update_traces(textposition='inside', textinfo='percent+label')
fig8.update_layout(height=500)
fig8.show()

: 

 ### 4.5 Análisis de Correlaciones y Relaciones

In [ ]:
# Matriz de correlación
variables_numericas = df[['monto_usd', 'num_oferentes', 'tiempo_proceso_dias', 'año']]
correlacion = variables_numericas.corr()

print("\nMatriz de Correlación:")
print(correlacion)

: 

In [ ]:
# Visualizar matriz de correlación
fig9 = px.imshow(
    correlacion,
    text_auto='.3f',
    title='Matriz de Correlación entre Variables Numéricas',
    labels=dict(color="Correlación"),
    color_continuous_scale='RdBu_r',
    aspect='auto',
    x=['Monto USD', 'Núm. Oferentes', 'Tiempo (días)', 'Año'],
    y=['Monto USD', 'Núm. Oferentes', 'Tiempo (días)', 'Año']
)
fig9.update_layout(height=500)
fig9.show()

: 

In [ ]:
# Gráfico de dispersión: Monto vs Tiempo de proceso
fig10 = px.scatter(
    df.sample(min(2000, len(df))),  # Muestra para mejor visualización
    x='tiempo_proceso_dias',
    y='monto_usd',
    color='tipo_compra',
    size='num_oferentes',
    title='Relación entre Tiempo de Proceso y Monto Contratado',
    labels={
        'tiempo_proceso_dias': 'Tiempo de Proceso (días)',
        'monto_usd': 'Monto (USD)',
        'tipo_compra': 'Tipo de Compra',
        'num_oferentes': 'Núm. Oferentes'
    },
    hover_data=['provincia', 'sector', 'año'],
    opacity=0.6
)
fig10.update_layout(height=600)
fig10.show()

: 

In [ ]:
# Análisis: Monto promedio según número de oferentes
oferentes_stats = df.groupby('num_oferentes').agg({
    'monto_usd': ['mean', 'median', 'count']
}).reset_index()
oferentes_stats.columns = ['num_oferentes', 'monto_promedio', 'monto_mediano', 'cantidad']
oferentes_stats = oferentes_stats[oferentes_stats['cantidad'] >= 10]  # Filtrar outliers

fig11 = go.Figure()
fig11.add_trace(go.Scatter(
    x=oferentes_stats['num_oferentes'],
    y=oferentes_stats['monto_promedio'],
    mode='lines+markers',
    name='Promedio',
    line=dict(width=3),
    marker=dict(size=10)
))
fig11.add_trace(go.Scatter(
    x=oferentes_stats['num_oferentes'],
    y=oferentes_stats['monto_mediano'],
    mode='lines+markers',
    name='Mediana',
    line=dict(width=3, dash='dash'),
    marker=dict(size=8)
))
fig11.update_layout(
    title='Monto según Número de Oferentes (Competencia)',
    xaxis_title='Número de Oferentes',
    yaxis_title='Monto (USD)',
    height=500,
    hovermode='x unified'
)
fig11.show()

: 

In [ ]:
# Box plot: Distribución de montos por tipo de compra
fig12 = px.box(
    df,
    x='tipo_compra',
    y='monto_usd',
    title='Distribución de Montos por Tipo de Compra (Box Plot)',
    labels={'tipo_compra': 'Tipo de Compra', 'monto_usd': 'Monto (USD)'},
    color='tipo_compra',
    log_y=True  # Escala logarítmica para mejor visualización
)
fig12.update_layout(height=500, xaxis_tickangle=-45, showlegend=False)
fig12.show()

: 

In [ ]:
# Análisis por tiempo de proceso
tiempo_bins = [0, 30, 60, 90, 120, 180]
tiempo_labels = ['0-30', '31-60', '61-90', '91-120', '121-180']
df['tiempo_categoria'] = pd.cut(df['tiempo_proceso_dias'], bins=tiempo_bins, labels=tiempo_labels)

tiempo_stats = df.groupby('tiempo_categoria').agg({
    'monto_usd': 'mean',
    'año': 'count'
}).reset_index()
tiempo_stats.columns = ['tiempo_categoria', 'monto_promedio', 'cantidad']

fig13 = px.bar(
    tiempo_stats,
    x='tiempo_categoria',
    y='cantidad',
    title='Distribución de Contratos según Tiempo de Proceso (días)',
    labels={'tiempo_categoria': 'Rango de Días', 'cantidad': 'Cantidad de Contratos'},
    color='monto_promedio',
    color_continuous_scale='Oranges',
    text='cantidad'
)
fig13.update_traces(textposition='outside')
fig13.update_layout(height=500)
fig13.show()

: 

 ### 4.6 Análisis Comparativo por Años

In [ ]:
# Comparación año a año
print("\n" + "="*80)
print("ANÁLISIS COMPARATIVO POR AÑOS (2015-2025)")
print("="*80)

for año in sorted(df['año'].unique()):
    df_año = df[df['año'] == año]
    print(f"\n--- AÑO {año} ---")
    print(f"  Total Contratos:     {len(df_año):,}")
    print(f"  Monto Total:         ${df_año['monto_usd'].sum():,.2f}")
    print(f"  Monto Promedio:      ${df_año['monto_usd'].mean():,.2f}")
    print(f"  Tiempo Promedio:     {df_año['tiempo_proceso_dias'].mean():.0f} días")
    print(f"  Sector Principal:    {df_año.groupby('sector').size().idxmax()}")
    print(f"  Provincia Principal: {df_año.groupby('provincia').size().idxmax()}")

: 

In [ ]:
# Gráfico comparativo: Métricas por año
fig14 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Cantidad de Contratos', 'Monto Total (USD)', 
                   'Monto Promedio (USD)', 'Tiempo Promedio (días)'),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

años = sorted(df['año'].unique())
cantidades = [len(df[df['año'] == año]) for año in años]
montos_totales = [df[df['año'] == año]['monto_usd'].sum() for año in años]
montos_promedios = [df[df['año'] == año]['monto_usd'].mean() for año in años]
tiempos_promedios = [df[df['año'] == año]['tiempo_proceso_dias'].mean() for año in años]

fig14.add_trace(go.Bar(x=años, y=cantidades, name='Cantidad', marker_color='lightblue'), row=1, col=1)
fig14.add_trace(go.Bar(x=años, y=montos_totales, name='Monto Total', marker_color='lightcoral'), row=1, col=2)
fig14.add_trace(go.Bar(x=años, y=montos_promedios, name='Monto Prom.', marker_color='lightgreen'), row=2, col=1)
fig14.add_trace(go.Bar(x=años, y=tiempos_promedios, name='Tiempo Prom.', marker_color='lightyellow'), row=2, col=2)

fig14.update_layout(height=700, showlegend=False, title_text="Comparativa de Métricas por Año")
fig14.show()

: 

 ## 5. Insights y Patrones Identificados

In [ ]:
# Identificar tendencias y patrones
print("\n" + "="*80)
print("INSIGHTS Y PATRONES CLAVE")
print("="*80)

# 1. Crecimiento interanual
años_sorted = sorted(df['año'].unique())
if len(años_sorted) > 1:
    primer_año = df[df['año'] == años_sorted[0]]
    ultimo_año = df[df['año'] == años_sorted[-1]]
    
    crecimiento_contratos = ((len(ultimo_año) - len(primer_año)) / len(primer_año)) * 100
    crecimiento_monto = ((ultimo_año['monto_usd'].sum() - primer_año['monto_usd'].sum()) / 
                        primer_año['monto_usd'].sum()) * 100
    
    print(f"\n1. CRECIMIENTO ({años_sorted[0]} - {años_sorted[-1]}):")
    print(f"   - Contratos: {crecimiento_contratos:+.1f}%")
    print(f"   - Monto Total: {crecimiento_monto:+.1f}%")

# 2. Concentración geográfica
total_contratos_pais = len(df)
top3_provincias = df.groupby('provincia').size().nlargest(3)
concentracion = (top3_provincias.sum() / total_contratos_pais) * 100

print(f"\n2. CONCENTRACIÓN GEOGRÁFICA:")
print(f"   - Top 3 provincias representan: {concentracion:.1f}% del total")
for prov, cant in top3_provincias.items():
    print(f"     • {prov}: {cant:,} contratos ({cant/total_contratos_pais*100:.1f}%)")

# 3. Competencia (número de oferentes)
oferentes_bajo = len(df[df['num_oferentes'] <= 3])
oferentes_alto = len(df[df['num_oferentes'] > 7])

print(f"\n3. NIVEL DE COMPETENCIA:")
print(f"   - Baja competencia (≤3 oferentes): {oferentes_bajo:,} ({oferentes_bajo/len(df)*100:.1f}%)")
print(f"   - Alta competencia (>7 oferentes): {oferentes_alto:,} ({oferentes_alto/len(df)*100:.1f}%)")

# 4. Eficiencia temporal
contratos_rapidos = len(df[df['tiempo_proceso_dias'] <= 45])
contratos_lentos = len(df[df['tiempo_proceso_dias'] > 120])

print(f"\n4. EFICIENCIA EN PROCESOS:")
print(f"   - Procesos rápidos (≤45 días): {contratos_rapidos:,} ({contratos_rapidos/len(df)*100:.1f}%)")
print(f"   - Procesos lentos (>120 días): {contratos_lentos:,} ({contratos_lentos/len(df)*100:.1f}%)")

# 5. Sectores prioritarios
top3_sectores_monto = df.groupby('sector')['monto_usd'].sum().nlargest(3)

print(f"\n5. SECTORES CON MAYOR INVERSIÓN:")
for sector, monto in top3_sectores_monto.items():
    pct = (monto / df['monto_usd'].sum()) * 100
    print(f"   • {sector}: ${monto:,.2f} ({pct:.1f}%)")

print("\n" + "="*80)

: 

 ## 6. Conclusiones y Recomendaciones

In [ ]:
print("\n" + "="*80)
print("CONCLUSIONES FINALES")
print("="*80)

conclusiones = f"""
1. VOLUMEN Y ESCALA:
   • Se analizaron {total_contratos:,} contratos públicos del período 2015-2025
   • Monto total: ${monto_total:,.2f} USD
   • Monto promedio por contrato: ${monto_promedio:,.2f} USD
   
2. TENDENCIAS TEMPORALES:
   • El análisis revela variaciones anuales en volumen y monto de contratación
   • Identificación de estacionalidad en ciertos meses del año
   • Patrones de crecimiento/decrecimiento según sectores
   
3. DISTRIBUCIÓN GEOGRÁFICA:
   • Alta concentración en las principales provincias (Pichincha, Guayas)
   • Oportunidad de descentralización en provincias con menor participación
   
4. EFICIENCIA Y COMPETENCIA:
   • Tiempo promedio de proceso: {tiempo_promedio:.0f} días
   • Correlación entre número de oferentes y eficiencia del proceso
   • Los procesos con mayor competencia tienden a ser más eficientes
   
5. SECTORES ESTRATÉGICOS:
   • Salud, Educación e Infraestructura concentran mayor inversión
   • Necesidad de equilibrio en inversión por sectores
   
RECOMENDACIONES:
   ✓ Implementar mecanismos para reducir tiempos de proceso
   ✓ Fomentar mayor competencia (más oferentes)
   ✓ Fortalecer transparencia en provincias con menor participación
   ✓ Establecer KPIs de seguimiento trimestral
   ✓ Desarrollar análisis predictivos para planificación presupuestaria
"""

print(conclusiones)
print("="*80)

: 

 ## 7. Exportación de Resultados

In [ ]:
# Guardar datasets procesados
df.to_csv('compras_publicas_limpio.csv', index=False, encoding='utf-8')
contratos_año.to_csv('analisis_por_año.csv', index=False)
provincias_stats.to_csv('analisis_por_provincia.csv', index=False)
sector_stats.to_csv('analisis_por_sector.csv', index=False)

print("\n✓ Archivos exportados correctamente:")
print("  - compras_publicas_limpio.csv")
print("  - analisis_por_año.csv")
print("  - analisis_por_provincia.csv")
print("  - analisis_por_sector.csv")

: 

 ## 8. Recursos Adicionales

 **Fuentes de datos:**
 - Portal de Datos Abiertos SERCOP: https://datosabiertos.compraspublicas.gob.ec/
 - Portal de Compras Públicas: https://www.compraspublicas.gob.ec/

 **Autor:** [Tu Nombre Completo]

 **Fecha:** Octubre 2025

 ---
 *Fin del Análisis*